# Hospital infrastructure in Canada — web interface

Course project: OSS 4008. This notebook downloads Statistics Canada hospital reference records, constructs a browser interface, and supports searches for Canadian cities and towns. Location references are not machine-learning predictions.

In [ ]:
%pip -q install "gradio>=5,<7" "folium>=0.19,<1" "pandas>=2,<3" "requests>=2,<3" "numpy>=1.24" "shapely>=2,<3" "pyproj>=3,<4"

## Load Canadian hospital reference records

Source: Statistics Canada ODHF v2 (September 2026). Run once each time Colab starts. If the public download is unavailable, the map and city search still work, but hospital reference markers are disabled. The dataset contains facility locations, not training annotations.

In [ ]:
from pathlib import Path
PREPARATION_SOURCE = '"""Fetch and normalize Statistics Canada\'s ODHF v2 GeoJSON for mapping.\n\nUses public data published September 24, 2026. Intended as a location reference only.\n"""\nfrom pathlib import Path\nimport json\nfrom io import BytesIO\nimport zipfile\nimport requests\nimport pandas as pd\nfrom pyproj import Transformer\nfrom shapely.geometry import shape\nfrom shapely.ops import transform\n\nSOURCE=\'https://www150.statcan.gc.ca/n1/pub/13-26-0001/2020001/zip/odhf_v2_geojson.zip\'\nTARGET=Path(\'/content/odhf_hospitals.csv\') if Path(\'/content\').exists() else Path(\'odhf_hospitals.csv\')\n\ndef is_hospital(properties):\n    # v2 field `type` documented by Statistics Canada.\n    kind=str(properties.get(\'type\') or \'\').strip().casefold()\n    return kind in (\'hospital\', \'hospitals\')\n\ndef build_dataset():\n    response=requests.get(SOURCE,timeout=100,headers={\'User-Agent\':\'OSS4008-Canada-Hospitals-CourseProject/1.0\'})\n    response.raise_for_status()\n    if len(response.content)>300_000_000:\n        raise ValueError(\'The dataset archive unexpectedly exceeded 300 MB.\')\n    records=[]\n    with zipfile.ZipFile(BytesIO(response.content)) as archive:\n        candidates=[n for n in archive.namelist() if n.lower().endswith((\'.geojson\',\'.json\')) and not n.startswith(\'__MACOSX/\')]\n        if not candidates:\n            raise ValueError(\'No GeoJSON file found in the ODHF v2 archive.\')\n        print(\'GeoJSON files:\', candidates)\n        for name in candidates:\n            # GeoJSON content may contain point or polygon geometries.\n            features=json.loads(archive.read(name)).get(\'features\') or []\n            for feature in features:\n                properties=feature.get(\'properties\') or {}\n                if not is_hospital(properties):\n                    continue\n                geometry=feature.get(\'geometry\')\n                if not geometry:\n                    continue\n                geom=shape(geometry)\n                if geom.is_empty:\n                    continue\n                point=geom.representative_point()\n                x,y=point.x,point.y\n                if abs(x)>180 or abs(y)>90:\n                    # National published projection: NAD83 / Statistics Canada Lambert (EPSG:3347).\n                    transformer=Transformer.from_crs(\'EPSG:3347\',\'EPSG:4326\',always_xy=True)\n                    x,y=transformer.transform(x,y)\n                if not (41<=y<=84 and -142<=x<=-52):\n                    continue\n                records.append({\'name\':str(properties.get(\'name\') or \'Unnamed hospital\'),\n                    \'latitude\':y,\'longitude\':x,\'type\':str(properties.get(\'type\') or \'\'),\n                    \'address\':str(properties.get(\'address\') or \'\'),\n                    \'province\':str(properties.get(\'prov_terr\') or \'\'),\n                    \'id\':str(properties.get(\'id\') or \'\'),\n                    \'geometry_type\':str(geometry.get(\'type\') or \'\')})\n    if not records:\n        raise ValueError(\'The archive contained no recognizable hospital features. Inspect the published field schema; no fake reference locations were generated.\')\n    df=pd.DataFrame(records).drop_duplicates(subset=[\'id\',\'name\',\'latitude\',\'longitude\'])\n    TARGET.parent.mkdir(parents=True,exist_ok=True)\n    df.to_csv(TARGET,index=False)\n    print(\'Saved\',len(df),\'hospital reference records to\',TARGET)\n    print(df[[\'name\',\'province\',\'geometry_type\']].head(8).to_string(index=False))\n    return df\n\nif __name__==\'__main__\':\n    build_dataset()\n'
Path("/content/prepare_odhf.py").write_text(PREPARATION_SOURCE,encoding="utf-8")
try:
    exec(compile(PREPARATION_SOURCE, "/content/prepare_odhf.py", "exec"), {"__name__":"__main__"})
except Exception as error:
    print("Reference data download unavailable:",type(error).__name__,str(error)[:500])
    print("The app will still launch, but without hospital markers.")

## Web application
The application and styling are maintained in `02_App/app.py`. The cell below creates the same module in the Colab runtime so you can run the notebook without mounting Google Drive.


In [ ]:
from pathlib import Path
APP_SOURCE = '"""Canada hospital infrastructure map — OSS4008 research application.\n\nReference layer: Statistics Canada ODHF v2.0 (2026).\nModel layer: uploaded GeoJSON produced by a separately validated detector.\nGeocoding: Open-Meteo Geocoding API, limited to deliberate user searches.\n"""\nfrom __future__ import annotations\n\nfrom functools import lru_cache\nimport html\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nimport folium\nfrom folium.plugins import Draw, Fullscreen, MarkerCluster\nimport gradio as gr\nimport pandas as pd\nimport requests\n\nDATA_FILE = Path(\'/content/odhf_hospitals.csv\') if Path(\'/content\').exists() else Path(\'odhf_hospitals.csv\')\nDATA_SOURCE = \'Statistics Canada — Open Database of Healthcare Facilities, v2.0 (2026)\'\nCANADA_CENTER = (56.1304, -106.3468)\nDEFAULT_CENTER = (44.6488, -63.5752)\nMAX_MARKERS = 900\n\n\ndef validate_coordinates(lat: float, lon: float, radius_km: float):\n    lat, lon, radius_km = float(lat), float(lon), float(radius_km)\n    if not all(math.isfinite(n) for n in (lat, lon, radius_km)):\n        raise ValueError(\'Coordinates must be finite numbers.\')\n    if not (41 <= lat <= 84 and -142 <= lon <= -52):\n        raise ValueError(\'Select a location within Canada.\')\n    if not (1 <= radius_km <= 100):\n        raise ValueError(\'Radius must be from 1 to 100 km.\')\n    return lat, lon, radius_km\n\n\n@lru_cache(maxsize=64)\ndef search_place(query: str):\n    """Search Canadian municipalities. Deliberate search only, no autocomplete."""\n    query = str(query).strip()\n    if len(query) < 2:\n        raise ValueError(\'Enter a Canadian city or town.\')\n    response = requests.get(\n        \'https://geocoding-api.open-meteo.com/v1/search\',\n        params={\'name\': query, \'count\': 10, \'language\': \'en\', \'countryCode\': \'CA\'},\n        timeout=12,\n        headers={\'User-Agent\': \'OSS4008-Canada-Hospitals-CourseProject/1.0\'},\n    )\n    response.raise_for_status()\n    matches = response.json().get(\'results\') or []\n    if not matches:\n        raise ValueError(f\'No Canadian location found for "{query}". Try including the province.\')\n    exact = [m for m in matches if m.get(\'name\', \'\').casefold() == query.split(\',\')[0].strip().casefold()]\n    chosen = (exact or matches)[0]\n    return (float(chosen[\'latitude\']), float(chosen[\'longitude\']),\n            f"{chosen.get(\'name\', query)}, {chosen.get(\'admin1\', \'Canada\')}")\n\n\n@lru_cache(maxsize=1)\ndef load_hospitals():\n    if not DATA_FILE.exists():\n        return pd.DataFrame(columns=[\'name\', \'latitude\', \'longitude\', \'type\', \'address\', \'province\']), \\\n               \'The Statistics Canada reference CSV has not been prepared. Run the dataset setup cell in the Colab notebook.\'\n    data = pd.read_csv(DATA_FILE)\n    for col in (\'name\', \'latitude\', \'longitude\'):\n        if col not in data:\n            raise ValueError(f\'Healthcare dataset missing column: {col}\')\n    data[\'latitude\'] = pd.to_numeric(data[\'latitude\'], errors=\'coerce\')\n    data[\'longitude\'] = pd.to_numeric(data[\'longitude\'], errors=\'coerce\')\n    data = data.dropna(subset=[\'latitude\', \'longitude\'])\n    data = data[data[\'latitude\'].between(41, 84) & data[\'longitude\'].between(-142, -52)]\n    return data, None\n\n\ndef haversine_km(lat, lon, latitudes, longitudes):\n    import numpy as np\n    p1 = np.radians(lat)\n    p2 = np.radians(latitudes.to_numpy(dtype=float))\n    dlat = p2 - p1\n    dlon = np.radians(longitudes.to_numpy(dtype=float) - lon)\n    a = np.sin(dlat / 2)**2 + np.cos(p1) * np.cos(p2) * np.sin(dlon / 2)**2\n    return 6371.0088 * 2 * np.arcsin(np.minimum(1, np.sqrt(a)))\n\n\ndef get_hospitals(lat, lon, radius_km):\n    data, error = load_hospitals()\n    if error or data.empty:\n        return data, error\n    selected = data.copy()\n    selected[\'distance_km\'] = haversine_km(lat, lon, selected.latitude, selected.longitude)\n    selected = selected[selected.distance_km <= radius_km].sort_values(\'distance_km\')\n    return selected, None\n\n\ndef read_predictions(path):\n    if not path:\n        return []\n    if Path(path).stat().st_size > 8_000_000:\n        raise ValueError(\'Prediction file exceeds 8 MB.\')\n    with open(path, encoding=\'utf-8\') as file:\n        obj = json.load(file)\n    if obj.get(\'type\') != \'FeatureCollection\':\n        raise ValueError(\'Prediction file must be a GeoJSON FeatureCollection.\')\n    features = obj.get(\'features\', [])\n    if len(features) > 3000:\n        raise ValueError(\'Prediction file exceeds 3,000 features.\')\n    points = []\n    for f in features:\n        geom = f.get(\'geometry\') or {}\n        if geom.get(\'type\') != \'Point\' or len(geom.get(\'coordinates\', [])) < 2:\n            continue\n        lon, lat = map(float, geom[\'coordinates\'][:2])\n        if not (-180 <= lon <= 180 and -90 <= lat <= 90):\n            continue\n        raw_conf = (f.get(\'properties\') or {}).get(\'confidence\')\n        confidence = float(raw_conf) if raw_conf is not None else None\n        if confidence is not None and not (0 <= confidence <= 1):\n            continue\n        points.append((lat, lon, confidence))\n    return points\n\n\ndef render_map(lat=DEFAULT_CENTER[0], lon=DEFAULT_CENTER[1], radius_km=25,\n               include_reference=True, predictions_file=None, location_label=\'Halifax, Nova Scotia\'):\n    lat, lon, radius_km = validate_coordinates(lat, lon, radius_km)\n    zoom = 11 if radius_km <= 30 else 9\n    map_obj = folium.Map(location=[lat, lon], zoom_start=zoom, tiles=\'CartoDB positron\', control_scale=True)\n    folium.TileLayer(\'OpenStreetMap\', name=\'Street map\').add_to(map_obj)\n    folium.Circle([lat, lon], radius=radius_km * 1000, color=\'#1763ae\',\n                  weight=2, fill=False, tooltip=\'Study area\').add_to(map_obj)\n    total = 0\n    error = None\n    if include_reference:\n        refs, error = get_hospitals(lat, lon, radius_km)\n        total = len(refs)\n        group = folium.FeatureGroup(name=\'Hospital references (Statistics Canada)\', show=True)\n        cluster = MarkerCluster().add_to(group)\n        for _, row in refs.head(MAX_MARKERS).iterrows():\n            title = html.escape(str(row[\'name\']))\n            address = html.escape(str(row.get(\'address\', \'\')))\n            folium.CircleMarker([float(row[\'latitude\']), float(row[\'longitude\'])], radius=5,\n                color=\'#b91c1c\', fill=True, fill_opacity=0.85,\n                tooltip=title, popup=f\'<strong>{title}</strong><p>{address}</p><small>Statistics Canada reference — not an AI prediction</small>\').add_to(cluster)\n        group.add_to(map_obj)\n    pred = read_predictions(predictions_file)\n    if pred:\n        layer = folium.FeatureGroup(name=\'Uploaded model predictions\', show=True)\n        for plat, plon, score in pred:\n            score_text = f\'{score:.2f}\' if score is not None else \'not provided\'\n            folium.CircleMarker([plat, plon],radius=7,color=\'#16803d\',fill=True,fill_opacity=0.9,\n                tooltip=f\'Model prediction — confidence {score_text}\').add_to(layer)\n        layer.add_to(map_obj)\n    Fullscreen().add_to(map_obj)\n    Draw(export=False, draw_options={\'polyline\':False,\'circle\':False,\'circlemarker\':False,\'marker\':False},\n         edit_options={\'edit\':False}).add_to(map_obj)\n    folium.LayerControl(collapsed=False).add_to(map_obj)\n    status = (f\'**{location_label}** · {total} hospital reference records within {radius_km:g} km · \'\n              f\'{len(pred)} uploaded model predictions.  \'\n              \'Reference records come from Statistics Canada; no model inference was performed.\')\n    if total > MAX_MARKERS:\n        status += f\' Showing the nearest {MAX_MARKERS} reference markers.\'\n    if error:\n        status += f\'  {error}\'\n    return map_obj._repr_html_(), status\n\n\ndef handle_search(place, radius, show_refs, prediction_file):\n    try:\n        lat, lon, name = search_place(place)\n        view, status = render_map(lat, lon, radius, show_refs, prediction_file, name)\n        return view, status, lat, lon, name\n    except Exception as exc:\n        return gr.skip(), f\'**Location search failed:** {type(exc).__name__}: {exc}\', gr.skip(), gr.skip(), gr.skip()\n\n\ndef handle_refresh(lat, lon, label, radius, show_refs, prediction_file):\n    try:\n        return render_map(lat, lon, radius, show_refs, prediction_file, label)\n    except Exception as exc:\n        return gr.skip(), f\'**Map update failed:** {type(exc).__name__}: {exc}\'\n\n\ndef assistant_reply(message, history, radius, show_refs, prediction_file):\n    """Handle simple navigation commands. Does not call an LLM or infer from imagery."""\n    history = list(history or [])\n    query = str(message or \'\').strip()\n    if not query:\n        return history, gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip(), \'\'\n    history.append({\'role\':\'user\',\'content\': query})\n    import re\n    place = re.sub(r\'^(please\\s+)?(find|show|locate|map|search|go to)(\\s+for)?\\s+(hospitals?\\s+)?(in|near|around|at)?\\s*\', \'\', query, flags=re.I).strip()\n    if not place or re.search(r\'^(help|what can you do|how does this work)\\??$\',query, re.I):\n        text = \'I can navigate the map to a Canadian city or town. Try “Find hospitals in Ottawa”. Hospital markers are Statistics Canada reference records, not detections from imagery.\'\n        history.append({\'role\':\'assistant\',\'content\':text})\n        return history, gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip(), \'\'\n    view, status, lat, lon, name = handle_search(place, radius, show_refs, prediction_file)\n    if isinstance(lat, (int,float)):\n        response = f\'Map moved to {name}. The markers show known hospital facilities within {radius:g} km. Image-based detection is not connected yet.\'\n    else:\n        response = f\'I could not find that Canadian location. Please try the town name and province. {status}\'\n    history.append({\'role\':\'assistant\',\'content\':response})\n    return history, view, status, lat, lon, name, \'\'\n\nCUSTOM_CSS = """\n/* Desktop application shell: keep the map, controls, and assistant in one viewport. */\n.gradio-container {max-width: 1800px !important; margin: 0 auto !important; padding: 9px 22px !important;}\n#page-title {margin: 1px 0 8px !important; padding: 0 !important;}\n#page-title h1 {font-size: 1.5rem !important; line-height: 1.35 !important; margin: 0 !important; letter-spacing: -.025em;}\n#main-layout {gap: 14px !important; align-items: stretch !important;}\n#map-pane {min-width: 0 !important; gap: 8px !important;}\n#search-group {padding: 6px !important; border-radius: 11px !important;}\n#search-fields {gap: 7px !important; align-items: center !important;}\n#search-fields > div:first-child {flex: 1 1 auto !important; min-width: 0 !important;}\n#place-field {min-width: 0 !important;}\n#place-field textarea, #place-field input {min-height: 38px !important;}\n#place-button {min-width: 96px !important; width: 110px !important; max-width: 116px !important; height: 40px !important; flex: 0 0 110px !important;}\n#map-box {overflow: hidden !important; min-height: 0 !important;}\n#map-box iframe {width: 100% !important; height: calc(100dvh - 255px) !important; min-height: 310px !important; border: none !important; border-radius: 10px !important; display: block !important;}\n#map-status {font-size: .76rem !important; opacity: .8; max-height: 36px; overflow-y: auto;}\n#map-status p {margin: 0 !important;}\n#quick-settings {padding: 4px 8px !important; gap: 10px !important; align-items: center !important; border-radius: 9px !important;}\n#quick-settings .wrap {gap: 5px !important;}\n#settings-label {min-width: 65px !important; flex: 0 0 auto !important;}\n#settings-label p {font-weight: 600; font-size: .86rem; margin: 0 !important;}\n#radius-control {min-width: 160px !important; flex: 1 1 auto !important;}\n#radius-control label {font-size: .79rem !important;}\n#reference-toggle {min-width: 150px !important; flex: 0 1 210px !important;}\n#apply-button {min-width: 75px !important; flex: 0 0 84px !important;}\n#assistant-panel {min-width: 285px !important; gap: 7px !important;}\n#assistant-heading {margin: 0 !important;}\n#assistant-heading h3 {margin: 0 !important; font-size: 1.04rem !important;}\n#assistant-chat {min-height: 0 !important;}\n#assistant-chat .message {font-size: .9rem !important;}\n#chat-compose {gap: 6px !important; align-items: center !important;}\n#chat-message {min-width: 0 !important;}\n#send-button {min-width: 70px !important; max-width: 84px !important; height: 40px !important;}\n#research-options {font-size: .8rem !important;}\n@media (min-width: 1024px) and (min-height: 650px) {\n  body {overflow-y: hidden !important;}\n  #assistant-chat {height: calc(100dvh - 172px) !important; max-height: calc(100dvh - 172px) !important;}\n}\n@media (max-width: 1023px), (max-height: 649px) {\n  body {overflow-y: auto !important;}\n  #map-box iframe {height: 440px !important;}\n  #assistant-chat {height: 320px !important;}\n  #quick-settings {flex-wrap: wrap !important;}\n}\n"""\n\nwith gr.Blocks(title=\'Hospital infrastructure in Canada\', fill_width=True, fill_height=True,\n               **({\'css\': CUSTOM_CSS, \'theme\': gr.themes.Soft()} if int(gr.__version__.split(\'.\')[0]) < 6 else {})) as demo:\n    gr.Markdown(\'# Hospital infrastructure in Canada\', elem_id=\'page-title\')\n    latitude = gr.State(DEFAULT_CENTER[0])\n    longitude = gr.State(DEFAULT_CENTER[1])\n    location_label = gr.State(\'Halifax, Nova Scotia\')\n    with gr.Row(equal_height=False, elem_id=\'main-layout\'):\n        with gr.Column(scale=7, elem_id=\'map-pane\'):\n            with gr.Group(elem_id=\'search-group\'):\n                with gr.Row(elem_id=\'search-fields\'):\n                    search_box = gr.Textbox(label=\'City or town\', show_label=False,\n                        placeholder=\'Search any Canadian city or town\', value=\'Halifax\',\n                        scale=8, container=False, elem_id=\'place-field\')\n                    search_button = gr.Button(\'Search\', variant=\'primary\', scale=0,\n                        min_width=100, elem_id=\'place-button\')\n            map_html = gr.HTML(elem_id=\'map-box\', container=False)\n            status = gr.Markdown(elem_id=\'map-status\')\n            with gr.Row(elem_id=\'quick-settings\'):\n                gr.Markdown(\'**Map settings**\', elem_id=\'settings-label\')\n                radius = gr.Slider(1, 100, value=25, step=1, label=\'Radius (km)\',\n                                   elem_id=\'radius-control\', container=False)\n                references = gr.Checkbox(value=True, label=\'Hospital markers\',\n                                          elem_id=\'reference-toggle\', container=False)\n                update = gr.Button(\'Apply\', variant=\'secondary\', elem_id=\'apply-button\',\n                                   min_width=70)\n            with gr.Accordion(\'Advanced · model prediction overlay\', open=False,\n                              elem_id=\'research-options\'):\n                predictions = gr.File(label=\'Import model output (GeoJSON)\',\n                                      file_types=[\'.geojson\', \'.json\'], type=\'filepath\')\n                gr.Markdown(\'Displays georeferenced predictions from a separately trained model. The app does not currently run image inference.\')\n        with gr.Column(scale=3, elem_id=\'assistant-panel\'):\n            gr.Markdown(\'### Map assistant\', elem_id=\'assistant-heading\')\n            chat = gr.Chatbot(label=\'Conversation\', height=415, elem_id=\'assistant-chat\',\n                value=[{\'role\':\'assistant\', \'content\':\'Where would you like to look? Try “Find hospitals in Ottawa”.\'}])\n            with gr.Row(elem_id=\'chat-compose\'):\n                command = gr.Textbox(label=\'Message\', show_label=False,\n                    placeholder=\'Ask to find a place…\', lines=1, container=False,\n                    elem_id=\'chat-message\', scale=6)\n                send = gr.Button(\'Send\', variant=\'primary\', scale=0,\n                                 min_width=74, elem_id=\'send-button\')\n    inputs = [radius, references, predictions]\n    search_button.click(handle_search, [search_box,*inputs], [map_html,status,latitude,longitude,location_label])\n    search_box.submit(handle_search, [search_box,*inputs], [map_html,status,latitude,longitude,location_label])\n    update.click(handle_refresh,[latitude,longitude,location_label,*inputs],[map_html,status])\n    chat_outputs = [chat,map_html,status,latitude,longitude,location_label,command]\n    send.click(assistant_reply,[command,chat,*inputs],chat_outputs)\n    command.submit(assistant_reply,[command,chat,*inputs],chat_outputs)\n    demo.load(lambda: render_map(),outputs=[map_html,status])\n\nif __name__ == \'__main__\':\n    demo.launch(**({\'css\': CUSTOM_CSS, \'theme\': gr.themes.Soft()} if int(gr.__version__.split(\'.\')[0]) >= 6 else {}))\n'
Path('/content/app.py').write_text(APP_SOURCE, encoding='utf-8')
print('Application source prepared:', len(APP_SOURCE), 'characters')
import importlib.util
spec = importlib.util.spec_from_file_location('canada_hospitals_app', '/content/app.py')
app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app)
print('Interface ready.')


## Start the application

Open the temporary Gradio URL produced below. Search for a city, town, or locality in Canada; press Enter or Search. Reference markers require successful preparation above.

In [ ]:
import gradio as gr
app.demo.launch(share=True, inline=False, debug=False, **({'css': app.CUSTOM_CSS, 'theme': gr.themes.Soft()} if int(gr.__version__.split('.')[0]) >= 6 else {}))


## Using the interface
- Search by city or town in the combined search bar above the map.
- Compact settings below the map control the radius and the hospital reference layer. Press Apply after changing settings.
- Use the right-hand map assistant for simple location navigation commands. It does not run a language model or analyze imagery.
- Advanced model output upload remains collapsed until trained predictions are available.
- Use a desktop browser at normal zoom for the single-screen layout; smaller windows can scroll as needed.
